#### Problem Set 7

In [1]:
import seaborn as sns

import matplotlib.pyplot as plt

import pandas as pd

import statsmodels.formula.api as smf

from patsy.contrasts import Treatment

In [2]:
from scipy.optimize import minimize

In [3]:
import numpy as np

##### Question 1

In [4]:

df = pd.read_csv("../data/pc_smoking.csv")

df['D'] = (df['state'] == 'CA')*1
df['t'] = (df['year'] - 1988) * df['D']

model = (smf.ols("""daily_packs ~ C(year) 
                                + C(state, Treatment(reference = 'CA')) 
                                + D 
                                + C(t, Treatment(reference = -1)):D""", 
                data = df)
         .fit(cov = 'robust')

        )

In [5]:
df_ref = pd.Series({"T": -1, "param": 0, "se": 0})

In [6]:
df_params = model.params.rename("param").reset_index()
df_se = model.HC0_se.rename("se").reset_index()
df_params = df_params[df_params['index'].apply(lambda y: ("C(t, Treatment(reference=-1))" in y) & (":D" in y))]
df_se = df_se[df_se['index'].apply(lambda y: ("C(t, Treatment(reference=-1))" in y) & (":D" in y))]
df_params['T'] = df_params['index'].str.replace("C(t, Treatment(reference=-1))[T.", "").str.replace("]:D", "").apply(int)
df_se['T'] = df_se['index'].str.replace("C(t, Treatment(reference=-1))[T.", "").str.replace("]:D", "").apply(int)
df_params = df_params.merge(df_se)
df_params = pd.concat([df_params, pd.DataFrame(df_ref).transpose()])
df_params['ub'] = df_params['param'] + 1.96 * df_params['se']
df_params['lb'] = df_params['param'] - 1.96 * df_params['se']

In [7]:
df_params = df_params.sort_values(by = 'T')

In [8]:
df_params[['T', 'param', 'se']]

,T,param,se
0,-18,23.002632,2.762700
1,-17,17.223684,2.764525
2,-16,14.407895,3.102545
3,-15,12.947368,2.864852
4,-14,12.118421,2.664182
5,-13,10.255263,2.652603
6,-12,6.826316,2.809932
7,-11,5.373684,2.353233
8,-10,5.713158,1.959536
9,-9,3.900000,1.788932


In [9]:
# Calculate error bars from confidence intervals
error_lower = df_params['param'] - df_params['lb']
error_upper = df_params['ub'] - df_params['param']
errors = [error_lower, error_upper]

plt.errorbar(df_params['T'], df_params['param'], 
             yerr=errors, 
             capsize=3, 
             capthick=1,
             marker='o', 
             markersize=4,
             linewidth=1.5)

plt.axhline(0, linestyle='--', color='gray', alpha=0.7)
plt.axvline(0, linestyle='--', color='gray', alpha=0.7)
plt.xlabel('Time (T)')
plt.ylabel('Parameter Estimate')
plt.savefig('event_study.png')
plt.close()

##### Question 2

In [10]:
df_sc = df.set_index(['state', 'year'])['daily_packs'].unstack().transpose()

In [11]:
START_YEAR = 1970
INTERVENTION_YEAR = 1989

In [12]:
STATE_TREATED = 'CA'

In [13]:
def normalize_weights(w):
    w_ = (np.exp(w)/np.sum(np.exp(w)))
    return w_

In [14]:
def get_loss(ZN, Zi, w, V = None):
    N_1 = Zi.shape[1]
    T0_1 = Zi.shape[0]
    w_ = normalize_weights(w).reshape([N_1,1])
    error = ZN - Zi@w_
    if V == None:
        V = np.eye(T0_1)
    loss = (error.T@V@error)
    return loss[0][0], w_

In [15]:
def get_sc_estimates(df_sc, year, state):
    
    ZN = df_sc[df_sc.index < year][[state]].to_numpy()
    Zi = df_sc[df_sc.index < year].drop([state], axis = 1).to_numpy()
    
    n_donors = Zi.shape[1]
    
    w0 = np.zeros(n_donors)

    result = minimize(lambda w: get_loss(ZN, Zi, w)[0], w0)
    
    w_opt = normalize_weights(result.x)
    df_w = pd.DataFrame(w_opt, index = df_sc.drop(state, axis = 1).columns, columns = ['w'])
    
    display("Weights")    
    display(df_w.apply(lambda y: np.round(y, 2)).sort_values(by = 'w', ascending = False).iloc[:10])
    
    sc = pd.DataFrame(df_sc.drop(state, axis = 1).to_numpy()@(df_w.to_numpy()), columns = [f'sc_{state}'], index = df_sc.index)
    
    sc[f'sc_{state}'].plot(label = 'Synthetic control')
    df_sc[state].plot(label = 'Observed')
    plt.axvline(year, linestyle = '--')
    plt.legend()
    plt.savefig(f'sc_{state}_{year}.png')

    plt.close()
    
    (df_sc[state] - sc[f'sc_{state}']).plot()
    plt.axvline(year, linestyle = '--')
    plt.axhline(0, linestyle = '--')
    plt.savefig(f'effect_{state}_{year}.png')
    plt.close()

    display("Policy impact estimates")    
    display(df_sc[state] - sc[f'sc_{state}'])

    

In [16]:
get_sc_estimates(df_sc, INTERVENTION_YEAR, STATE_TREATED)

'Weights'

,w
state,
UT,0.39
MT,0.25
NV,0.21
CT,0.11
NH,0.04
SC,0.00
NM,0.00
OH,0.00
OK,0.00


'Policy impact estimates'

year
1970     5.587179
1971     1.173682
1972    -1.063730
1973     0.173570
1974     0.237793
1975     0.291521
1976    -0.425767
1977    -0.658164
1978    -0.111330
1979    -0.983137
1980    -0.033987
1981    -1.011669
1982    -1.124156
1983    -0.195447
1984     1.398090
1985    -0.399778
1986     0.107348
1987    -2.957823
1988    -1.958691
1989    -8.583347
1990    -9.366725
1991   -12.656785
1992   -13.810581
1993   -17.554094
1994   -22.068576
1995   -22.970522
1996   -24.072296
1997   -26.331361
1998   -23.498578
1999   -27.604663
2000   -26.696062
dtype: float64

In [17]:
get_sc_estimates(df_sc, 1982, STATE_TREATED)

'Weights'

,w
state,
CT,0.52
ID,0.38
NV,0.10
AL,0.00
SD,0.00
NM,0.00
OH,0.00
OK,0.00
PA,0.00


'Policy impact estimates'

year
1970     2.786655
1971    -0.361201
1972    -1.753281
1973     1.270235
1974     0.195837
1975     2.566958
1976     1.487355
1977    -1.069001
1978    -0.266996
1979    -2.046422
1980    -2.638202
1981    -2.412335
1982    -3.068535
1983    -6.696233
1984    -6.717048
1985    -7.833678
1986    -7.708825
1987   -10.031787
1988   -10.693070
1989   -13.494836
1990   -17.694417
1991   -20.353203
1992   -19.274257
1993   -21.436166
1994   -27.022080
1995   -24.617683
1996   -22.864702
1997   -23.705609
1998   -27.141638
1999   -29.854782
2000   -30.252617
dtype: float64

In [18]:
get_sc_estimates(df_sc, 1995, STATE_TREATED)

'Weights'

,w
state,
NM,0.88
NH,0.12
AL,0.00
SC,0.00
NV,0.00
OH,0.00
OK,0.00
PA,0.00
RI,0.00


'Policy impact estimates'

year
1970    11.047691
1971     5.235753
1972    -0.401080
1973     2.997945
1974     5.222331
1975     3.259628
1976     2.098409
1977     1.960231
1978     2.197157
1979     1.708909
1980    -0.629084
1981    -2.191741
1982     0.120753
1983    -2.567078
1984     0.132319
1985     0.669059
1986    -1.956254
1987     1.106542
1988    -0.431544
1989    -4.306787
1990    -3.195268
1991   -10.558156
1992   -12.933307
1993   -15.582866
1994   -20.295036
1995   -22.032193
1996   -22.732147
1997   -22.068470
1998   -24.193551
1999   -26.493504
2000   -23.882077
dtype: float64

##### Question 3

In [19]:
df['year'] = df['year'].apply(int)

In [20]:
df_reg = df.set_index(['state', 'year'])['daily_packs'].unstack().transpose()

In [21]:
model_v = smf.ols('CA ~ AL + AR + CO + CT + DE + GA + IA + ID + IL + IN - 1', data = df_reg.loc[df_reg.index < INTERVENTION_YEAR]).fit()

In [22]:
model_v.predict(df_reg).plot()
df_reg['CA'].plot()
plt.axvline(INTERVENTION_YEAR, linestyle = '--')
plt.savefig("pred_q3.png")
plt.close()

In [23]:
(df_reg['CA'] - model_v.predict(df_reg)).plot()
plt.axvline(INTERVENTION_YEAR, linestyle = '--')
plt.axhline(0, linestyle = '--')
plt.savefig("effect_q3.png")
(model_v.predict(df_reg) - df_reg['CA'])
plt.close()


In [24]:
df_params_v = pd.DataFrame([df_reg.drop("CA", axis = 1).columns, model_v.params]).transpose()
df_params_v.columns = ['state', 'w']
df_params_v

,state,w
0,AL,0.700429
1,AR,-0.874537
2,CO,0.2012
3,CT,0.567806
4,DE,0.066191
5,GA,-0.294394
6,IA,0.153699
7,ID,0.096786
8,IL,0.267217
9,IN,0.115484


##### Question 4

In [25]:
model_v_r = (smf.ols(f'CA ~ {" + ".join(df_reg.drop("CA", axis = 1).columns)} - 1', 
                     data = df_reg.loc[df_reg.index < INTERVENTION_YEAR])
             .fit_regularized(L1_wt=0.)
            )

In [26]:
df_params_v_reg = pd.DataFrame([df_reg.drop("CA", axis = 1).columns, model_v_r.params]).transpose()
df_params_v_reg.columns = ['state', 'w']
df_params_v_reg

,state,w
0,AL,-0.026739
1,AR,0.035077
2,CO,0.11831
3,CT,0.167916
4,DE,-0.033909
5,GA,-0.0126
6,IA,0.022378
7,ID,0.04372
8,IL,0.08647
9,IN,-0.095555


In [27]:
model_v_r.predict(df_reg)[1989]

87.72539345936787

In [28]:
model_v_r.predict(df_reg).plot()
df_reg['CA'].plot()
plt.axvline(INTERVENTION_YEAR, linestyle = '--')
plt.savefig("pred_q4.png")
plt.close()

In [29]:
(df_reg['CA'] - model_v_r.predict(df_reg)).plot()
plt.axvline(INTERVENTION_YEAR, linestyle = '--')
plt.axhline(0, linestyle = '--')
plt.savefig("effect_q4.png")
(df_reg['CA'] - model_v_r.predict(df_reg))
plt.close()

##### Question 5

In [30]:
df_years = df_reg.transpose()
df_years = df_years.rename(columns = lambda y: f"year_{y}")

In [31]:
model_h_r = (smf.ols(f"""year_{INTERVENTION_YEAR} ~ {" + ".join(map(lambda y: f'year_{y}', range(START_YEAR, INTERVENTION_YEAR)))} - 1""",
                     data = df_reg[df_reg.index < INTERVENTION_YEAR + 1].transpose().drop("CA").rename(columns = lambda y: f"year_{y}"))
             .fit_regularized(L1_wt=0.)
            )

In [32]:
df_params_h_reg = pd.DataFrame([df_years.columns, model_h_r.params]).transpose()
df_params_h_reg.columns = ['year', 'w']
df_params_h_reg

,year,w
0,year_1970,-0.05806
1,year_1971,0.236009
2,year_1972,-0.335926
3,year_1973,0.289561
4,year_1974,-0.137833
5,year_1975,-0.029246
6,year_1976,-0.686771
7,year_1977,0.719754
8,year_1978,-0.12053
9,year_1979,0.416174


In [33]:
df_years['year_1989']['CA']

82.4

In [34]:
model_h_r.predict(df_years)['CA']

87.72539345936877

In [35]:
df_years['year_1989']['CA'] - model_h_r.predict(df_years)['CA']

-5.325393459368769

##### Question 6

In [36]:
get_sc_estimates(df_sc, INTERVENTION_YEAR, 'TX')

'Weights'

,w
state,
ND,0.24
OK,0.24
PA,0.18
MN,0.17
AR,0.09
CT,0.08
RI,0.00
NH,0.00
NM,0.00


'Policy impact estimates'

year
1970     2.247899
1971    -0.091985
1972     0.196235
1973    -2.013617
1974    -1.351613
1975    -2.936542
1976    -2.996663
1977    -0.794469
1978    -0.210053
1979     0.343982
1980     2.384202
1981     0.366148
1982     2.334901
1983     1.647925
1984     0.538073
1985     1.094419
1986     2.655398
1987    -1.582734
1988    -3.917857
1989    -2.708025
1990    -5.225603
1991   -12.457825
1992   -10.935120
1993    -6.797725
1994   -13.049287
1995   -18.393675
1996   -18.103955
1997   -19.686378
1998   -19.317639
1999   -22.262826
2000   -17.616431
dtype: float64

In [37]:
get_sc_estimates(df_sc, 1982, 'TX')

'Weights'

,w
state,
AR,0.49
CT,0.16
OK,0.13
PA,0.12
IA,0.07
OH,0.02
MN,0.01
AL,0.00
RI,0.00


'Policy impact estimates'

year
1970     0.068657
1971     0.102609
1972    -0.004332
1973    -0.616380
1974     0.665223
1975    -0.883966
1976     0.287059
1977     0.225838
1978    -0.614053
1979     0.030422
1980     0.375403
1981     0.446739
1982     4.509965
1983     0.356425
1984    -3.059073
1985    -4.913416
1986    -5.575066
1987   -10.266090
1988   -16.665834
1989   -15.091226
1990   -18.132298
1991   -25.293830
1992   -30.614607
1993   -19.315196
1994   -21.481774
1995   -28.218840
1996   -27.032584
1997   -28.151594
1998   -27.785577
1999   -30.076725
2000   -24.559832
dtype: float64

In [38]:
get_sc_estimates(df_sc, 1995, 'TX')

'Weights'

,w
state,
ND,0.34
CT,0.32
OK,0.27
MN,0.05
CA,0.01
RI,0.00
NH,0.00
NM,0.00
NV,0.00


'Policy impact estimates'

year
1970    -0.690512
1971    -1.519541
1972    -2.197316
1973    -2.763324
1974    -2.039398
1975    -3.273515
1976    -3.304259
1977    -1.159585
1978     0.327208
1979     0.987779
1980     3.319534
1981     1.740471
1982     3.322291
1983     3.457135
1984     1.701172
1985     3.986997
1986     5.379668
1987     1.565966
1988    -1.183927
1989     1.039432
1990     0.087734
1991    -6.650427
1992    -3.797195
1993    -1.013572
1994    -7.108462
1995   -12.729423
1996   -12.543161
1997   -13.761891
1998   -13.628388
1999   -16.436638
2000   -12.554310
dtype: float64